# Tutorial: Hybrid Semantic Memory Walkthrough

Audience:
- Engineers integrating the package into YClientReddit or YClient.

Prerequisites:
- Basic Python packaging knowledge.
- Familiarity with dataclasses and runtime adapters.

Learning goals:
- Build a `hybrid_semantic` engine.
- Record interactions and inspect reply context.
- Understand what the backend stores and returns.


## Outline

1. Setup the import path and runtime stub
2. Create a `hybrid_semantic` engine
3. Record memory events
4. Inspect reply, browse, and style context


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from yclient_memory import build_memory_engine
from yclient_memory.config import MemoryConfig
from yclient_memory.contracts import CommentMemoryEvent, PostMemoryEvent, PostStyleRequest, ReplyMemoryRequest

class RuntimeStub:
    def llm_json(self, prompt_key, variables, config=None):
        return {}

    def llm_text(self, prompt_key, variables, config=None):
        return ''

    def get_author_id_and_username(self, post_id):
        return 55, 'alice'

    def get_thread_root_id(self, post_id):
        return 900

    def get_recent_root_posts(self, round_id, limit=24, rounds_back=18):
        return []

    def get_post_text(self, post_id):
        return 'that benchmark evidence is weak'

    def persona_snapshot(self):
        return {'persona': 'skeptical forum agent'}

    def decision_log(self, payload):
        pass

runtime = RuntimeStub()
config = MemoryConfig.from_mapping({'memory_backend': 'hybrid_semantic', 'memory_vote_signal_only': True})
engine = build_memory_engine(backend='hybrid_semantic', config=config, runtime=runtime)
engine.capabilities()


## Step 1 - Seed the backend with a post and a conversation

The next cell records a root post plus one comment interaction. That is enough to populate the social card, thread card, and lexical event store used by reply-context assembly.


In [ ]:
engine.record_post(PostMemoryEvent(round_id=1, post_id=1, user_id=7, text='TITLE: benchmark drama

people keep arguing about evidence quality.'))
engine.record_comment(
    CommentMemoryEvent(
        round_id=2,
        target_post_id=11,
        thread_root_id=900,
        other_user_id=55,
        other_username='alice',
        other_text='that benchmark is weak',
        my_text='nah your benchmark evidence is weak',
        conv_text='@alice benchmark takes again',
    )
)
reply = engine.build_reply_context(
    ReplyMemoryRequest(
        query_text='compose a reply about the benchmark argument',
        other_user_id=55,
        other_username='alice',
        thread_root_id=900,
        round_id=2,
    )
)
{
    'reply_context': reply.rendered_text,
    'cues': reply.cues.rendered_text,
    'continuity_text': reply.continuity_text,
}


## Exercises

- Toggle `memory_vote_signal_only` to `False` and inspect the stored vote artifact behavior.
- Add more root posts and call `build_post_style_context()` again once the digest becomes mature.
- Trigger `maintenance_tick()` after enough events to observe community-digest or reflection updates.


In [ ]:
style = engine.build_post_style_context(PostStyleRequest(round_id=2))
{'style_context': style.rendered_text, 'style_usage': style.diagnostics.usage}
